In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
#1. Load Data
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(DATA_URL)

In [3]:
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# 2. Baseline Model (Without Feature Engineering)
X_base = df.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df['Survived']

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

num_features_base = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features_base = ['Pclass', 'Sex', 'Embarked']

In [6]:
# Numerical Pipeline: Impute missing values with median, then scale
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [7]:
# Categorical Pipeline: Impute missing values with most frequent, then one-hot encode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [8]:
# Combine via ColumnTransformer
preprocessor_base = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features_base),
        ('cat', cat_transformer, cat_features_base)
    ]
)

In [9]:
# Complete Pipeline (Preprocessing + Model)
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_base),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [10]:
# Fit & Evaluate Baseline
baseline_pipeline.fit(X_train_base, y_train)
y_pred_base = baseline_pipeline.predict(X_test_base)

base_acc = accuracy_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base)

In [11]:
print("=== BASELINE PIPELINE RESULTS ===")
print(f"Accuracy: {base_acc:.4f}")
print(f"F1-Score: {base_f1:.4f}\n")

=== BASELINE PIPELINE RESULTS ===
Accuracy: 0.8156
F1-Score: 0.7442



In [13]:
# 3. Feature Engineering & Evaluation
df_fe = df.copy()

# Feature 1: FamilySize = Siblings/Spouses + Parents/Children + 1 (self)
df_fe['FamilySize'] = df_fe['SibSp'] + df_fe['Parch'] + 1

# Feature 2: FarePerPerson = Total Fare / FamilySize
df_fe['FarePerPerson'] = df_fe['Fare'] / df_fe['FamilySize']

X_fe = df_fe.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])

X_train_fe, X_test_fe, _, _ = train_test_split(
    X_fe, y, test_size=0.2, random_state=42, stratify=y
)

num_features_fe = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'FarePerPerson']
cat_features_fe = ['Pclass', 'Sex', 'Embarked']

In [14]:
# Updated Preprocessor
preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features_fe),
        ('cat', cat_transformer, cat_features_fe)
    ]
)

In [15]:
# New Pipeline
fe_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [16]:
# Fit & Evaluate Engineered Pipeline
fe_pipeline.fit(X_train_fe, y_train)
y_pred_fe = fe_pipeline.predict(X_test_fe)

fe_acc = accuracy_score(y_test, y_pred_fe)
fe_f1 = f1_score(y_test, y_pred_fe)

print("=== FEATURE ENGINEERED PIPELINE RESULTS ===")
print(f"Accuracy: {fe_acc:.4f}")
print(f"F1-Score: {fe_f1:.4f}\n")

=== FEATURE ENGINEERED PIPELINE RESULTS ===
Accuracy: 0.8045
F1-Score: 0.7407



In [17]:
# 4. Save Final Pipeline
MODEL_FILENAME = "titanic_pipeline.joblib"
joblib.dump(fe_pipeline, MODEL_FILENAME)
print(f"Saved pipeline successfully to '{MODEL_FILENAME}'.")

# Verification: Reload and test single prediction
loaded_pipeline = joblib.load(MODEL_FILENAME)
sample_score = loaded_pipeline.score(X_test_fe, y_test)
print(f"Loaded model validation accuracy: {sample_score:.4f}")

Saved pipeline successfully to 'titanic_pipeline.joblib'.
Loaded model validation accuracy: 0.8045
